## Configuration

In [ ]:
from pathlib import Path
import tomllib
from utils.utils import *
import json

PATH_EEG = Path("../EEG/EEG/results/")
FILENAME = 'Case_06_ExG.csv'
MARKER_FILE = 'Case_06_Marker.csv'
CONFIG_PATH = "../EEG/EEG/config/config.toml"

with open(CONFIG_PATH, "rb") as f:
    config = tomllib.load(f)

BANDS = config['BANDS']
SEGMENT_DEF = config['SEGMENT_DEF']
RANDOM_SEED = config['RANDOM_SEED']
DURATION = config['DURATION']

SFREQ = config['SFREQ']
L_FREQ = config['L_FREQ']
H_FREQ = config['H_FREQ']

## Raw Data

In [ ]:
raw_full, df_full, df_markers_full = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

raws, labels = extract_segments(raw_full, df_full, df_markers_full, SEGMENT_DEF)

In [ ]:
plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Theta')

## Preprocessing

In [ ]:
import mne
from mne.preprocessing import ICA
raw, df, df_markers = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

# ICA
# n_components=15 (16 Channels, ICA finding max n_channels - 1 or same number)
ica = ICA(n_components=15, max_iter='auto', random_state=RANDOM_SEED)

print(">>> ICA (Might take a while)...")
ica.fit(raw)

print(">>> Showing Components...")
ica.plot_components()
plt.show()

ica.plot_sources(raw, show_scrollbars=True)
plt.show()

In [ ]:
exclude_indices = [0, 1, 2, 5, 6, 7]

print(f">>> Removing ingredients: {exclude_indices}")
ica.exclude = exclude_indices

raw_clean = raw.copy()
ica.apply(raw_clean)

print(">>> Cleaned data.")

target_chan = "F3" if "F3" in raw.ch_names else raw.ch_names[0]

print(f"Comparison on the channel {target_chan}...")
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(
    raw.get_data(picks=target_chan)[0, 1000:2500],
    color="red",
    alpha=0.5,
    label="Before cleaning (Original)",
)
ax.plot(
    raw_clean.get_data(picks=target_chan)[0, 1000:2500],
    color="black",
    label="After cleaning (Clean)",
)
ax.set_title(f"Artifact Removal Effect (Eyes and Muscles) - Channel {target_chan}")
ax.legend()
plt.show()

In [ ]:
raws, labels = extract_segments(raw_clean, df_full, df_markers_full, SEGMENT_DEF)

plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Gamma')

#### Analiza Pasma Alpha (8–12 Hz):
W trybie Brainrot (szczególnie pasywnym) widać silną ciemnoczerwoną plamę w lewym dolnym rogu (obszar O1). Czerwień w Alfie oznacza uśpienie/wyłączenie tego obszaru. Reszta mózgu jest mocno wybudzona (granatowa).

Kiedy pacjent przechodzi do treści Smart (zarówno pasywnie, jak i ze scrollem), ta czerwona plama znika – cały lewy tył głowy robi się granatowy (w pełni wybudzony).

Bierne oglądanie "głupotek" dosłownie odcina jego lewą korę wzrokowo-językową. Dopiero konieczność przetwarzania mądrzejszych treści (Smart) "budzi" tę analityczną część mózgu do życia.

#### Analiza Pasma Beta (13–30 Hz):
W trybie pasywnego Brainrotu (lewa góra), dokładnie w tym samym miejscu, które "spało" w Alfie (O1), widać ciemnoczerwoną plamę wysokiej aktywności Beta. We wszystkich innych trybach Beta na O1 niemal zanika.

To dowód na wysiłek. Gdy pacjent patrzy na chaotyczne filmiki, jego lewa półkula potyliczna próbuje za wszelką cenę "skanować" ekran. Pracuje mocno, że system zaczyna się w tym miejscu awaryjnie wyłączać.

#### Analiza Pasma Delta (1–4 Hz):
Wysoka Delta oznacza ostre znużenie poznawcze lub stan "awaryjnego uśpienia" kory mózgowej w danym rejonie.

Ten pacjent wykazuje bardzo silne, punktowe wyczerpanie w okolicach lewej kory potylicznej (O1) w dwóch skrajnie różnych trybach. Czerwona plama jest tam gęsta zarówno w trybie pasywnego Brainrotu, jak i podczas Smart Scroll.

To oznacza, że lewy analizator wzrokowy tego pacjenta jest podatny na "przepalenie", ale z dwóch różnych powodów:

* W Brainrocie: O1 wchodzi w stan "szoku" i awaryjnie się odcina.

* W Smart Scroll: Tutaj bodźce są wolniejsze, ale konieczność skupienia się na dłuszej treści to dla pacjenta wysiłek analityczny. Lewa kora, odpowiadająca za czytanie, przemęcza się.

#### Analiza Pasma Gamma (>30 Hz):
Lewa potylica (O1) płonie na czerwono z powodu wysokiej Gammy. W trybach Smart Gamma jest o wiele niższa.

Gamma odpowiada m.in. za integrację sensoryczną (visual binding). Pędzący montaż krótkich form wideo zmusza mózg tego pacjenta do pracy, by "skleić" treść w jedną, sensowną całość.

## Spektogram

In [ ]:
# Alpha (fatigue/relax)
plot_band_power_over_time(raws, labels, band=(8, 12), band_name='Alpha')
# Beta (Focus)
plot_band_power_over_time(raws, labels, band=(13, 30), band_name='Beta')

In [ ]:
# Emotion
calculate_asymmetry(raws, labels, ch_left='F3', ch_right='F4')

# According to Heller's model, parietal asymmetry is responsible for physiological arousal
calculate_asymmetry(raws, labels, ch_left='P3', ch_right='P4')

# Image processing method.
calculate_asymmetry(raws, labels, ch_left='O1', ch_right='O2')

# Motor
calculate_asymmetry(raws, labels, ch_left='C3', ch_right='C4')

### Podsumowanie

#### Kora Czołowa (F4 vs F3)
Wszystkie wskaźniki czołowe są dodatnie.

Mimo że układ wzrokowy jest przemęczony, ten pacjent nie odczuwa z tego powodu żadnego dyskomfortu ani frustracji. Dodatnie FAA to sygnatura motywacji dążeniowej i pozytywnego afektu.Największy skok zaangażowania widzać w fazie Brainrot Scroll – fizyczna interakcja z szybko zmieniającym się wideo daje najwyższy zastrzyk dopaminy i chęci do działania w całym badaniu.

#### Kora Ciemieniowa (P4 vs P3)
Wszystkie słupki są na plusie.

Brak aktywności na prawej ciemieniówce (P4) oznacza, że pacjent nie odczuwa żadnego stresu przestrzennego ani zagubienia w interfejsie. Najbardziej zrelaksowany przestrzennie jest podczas Smart Scroll. Kiedy czyta, jego lewa kora ciemieniowa (P3 - odpowiedzialna m.in. za integrację mowy i logiki) pracuje najmocniej, a prawa odpoczywa.

#### Kora Potyliczna (O2 vs O1)
Wszystkie słupki są na minusie.

Wynik ujemny oznacza, że na lewej półkuli potylicznej (O1) występuje siln moc fali Alpha (stan wyłączenia/snu).

Lewy, analityczny procesor wzrokowy pacjenta po prostu "leży". Przez to, że O1 odmawia posłuszeństwa, całą pracę musi przejmować prawa kora wzrokowa (O2), co zmusza go do czysto obrazowego, "holistycznego" patrzenia na ekran, mimo że lewa strona desperacko (i bezskutecznie) próbuje analizować detale.

#### Kora Ruchowa (C4 vs C3)
Słupki są potężnie dodatnie (od 1.5 do ponad 2.0).

Skrajna dominacja lewej kory ruchowej (C3). Pacjent przewija prawym kciukiem w sposób niezwykle pewny, automatyczny i zdecydowany. To nie jest nerwowe "klikanie" – jego układ motoryczny działa jak dobrze naoliwiona maszyna, całkowicie oddzielona od przeciążonego wzroku.
